# # Artwork Clustering for Similarity Matching with ResNet50 (Jupyter Notebook)

# ## 1. Setup and Imports

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Lambda
import tensorflow.keras.backend as K
import numpy as np
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import datetime
import shutil

# ## 2. Helper Functions

In [2]:
def create_base_network(input_shape):
    """Creates the base (shared) ResNet50 network."""
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

    # Initially freeze all layers
    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)  # Add a dense layer
    x = Lambda(lambda x: K.l2_normalize(x, axis=1))(x)  # L2 normalization
    return Model(base_model.input, x)

def contrastive_loss(y_true, y_pred, margin=1.0):
    """Contrastive loss function."""
    square_pred = K.square(y_pred)
    margin_square = K.square(K.maximum(margin - y_pred, 0))
    loss = K.mean(y_true * square_pred + (1 - y_true) * margin_square)
    return loss

def accuracy(y_true, y_pred):
    '''Compute classification accuracy with a fixed threshold on distances.'''
    return K.mean(K.equal(y_true, K.cast(y_pred < 0.5, y_true.dtype)))

def load_images_from_directory(directory, input_shape):
    """Loads, preprocesses, and returns images and filenames from a directory."""
    images = []
    filenames = []
    for filename in os.listdir(directory):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            try:
                img_path = os.path.join(directory, filename)
                img = load_img(img_path, target_size=input_shape[:2])
                img_array = img_to_array(img)
                img_array = preprocess_input(img_array)
                images.append(img_array)
                filenames.append(filename)
            except Exception as e:
                print(f"Error loading image {filename}: {e}")
    return np.array(images), filenames

# ## 3. Model Definition

In [ ]:
def find_closest_cluster(embedding, cluster_centers):
    distances = np.linalg.norm(cluster_centers - embedding, axis=1)
    closest_cluster_index = np.argmin(distances)
    return closest_cluster_index, distances[closest_cluster_index]

def save_cluster_images(lostart_dir, lostart_filenames, lostart_cluster_labels, output_base_dir, max_images_per_cluster=50):
    clusters_dir = os.path.join(output_base_dir, "Clusters")
    os.makedirs(clusters_dir, exist_ok=True)

    for cluster_index in range(max(lostart_cluster_labels) + 1):  # Iterate through all clusters
        cluster_dir = os.path.join(clusters_dir, f"Cluster_{cluster_index}")
        os.makedirs(cluster_dir, exist_ok=True)

        image_indices = np.where(lostart_cluster_labels == cluster_index)[0]
        count = 0
        for i in image_indices:
            if count >= max_images_per_cluster:
                break
            src_path = os.path.join(lostart_dir, lostart_filenames[i])
            dst_path = os.path.join(cluster_dir, lostart_filenames[i])
            try:
                shutil.copy2(src_path, dst_path)  # Copy the image file
                count += 1
            except Exception as e:
                print(f"Error copying image {lostart_filenames[i]}: {e}")

def save_query_results(all_dir, all_filenames, all_embeddings,
                       lostart_dir, lostart_filenames, lostart_cluster_labels,
                       cluster_centers, output_base_dir, max_images_per_result=10):
    """Saves query results (query + closest cluster images)."""

    results_dir = os.path.join(output_base_dir, "Results")
    os.makedirs(results_dir, exist_ok=True)

    for i, all_embedding in enumerate(all_embeddings):
        closest_cluster_index, distance = find_closest_cluster(all_embedding, cluster_centers)
        query_filename = all_filenames[i]
        query_image_path = os.path.join(all_dir, query_filename)

        # Create a directory for this query result
        query_result_dir = os.path.join(results_dir, f"Query_{i}_{os.path.splitext(query_filename)[0]}")
        os.makedirs(query_result_dir, exist_ok=True)

        # 1. Save the query image
        try:
            query_img = load_img(query_image_path) # Load *without* resizing
            query_img_save_path = os.path.join(query_result_dir, f"Query_{query_filename}")
            query_img.save(query_img_save_path)
            # Create and save a plot with the image and title
            plt.figure(figsize=(6, 6))
            plt.imshow(query_img)
            plt.title(f"Query Image: {query_filename} (Closest Cluster: {closest_cluster_index}, Distance: {distance:.4f})")
            plt.axis('off')
            plt.savefig(os.path.join(query_result_dir, f"Query_{os.path.splitext(query_filename)[0]}_plot.jpg"))  # Save the plot
            plt.close()

        except Exception as e:
             print(f"Error saving or plotting query image {query_filename}: {e}")


        # 2. Save closest cluster images (up to max_images_per_result)
        cluster_image_indices = np.where(lostart_cluster_labels == closest_cluster_index)[0]
        for j, image_index in enumerate(cluster_image_indices):
            if j >= max_images_per_result:
                break
            try:
                cluster_filename = lostart_filenames[image_index]
                src_path = os.path.join(lostart_dir, cluster_filename)
                dst_path = os.path.join(query_result_dir, f"Cluster_{closest_cluster_index}_{cluster_filename}")
                shutil.copy2(src_path, dst_path)
            except Exception as e:
                print(f"Error copying cluster image {cluster_filename}: {e}")



In [4]:
input_shape = (224, 224, 3)
lostart_dir = "images/lostart"
all_dir = "images/all"

# Create output directory with timestamp
current_date = datetime.datetime.now().strftime("%Y_%m_%d")
output_base_dir = os.path.join("Exports", f"{current_date}-Untrained Model First Run")
os.makedirs(output_base_dir, exist_ok=True)  # Ensure the base directory exists

# Create the base network (feature extractor)
base_network = create_base_network(input_shape)

# Load and embed images
lostart_images, lostart_filenames = load_images_from_directory(lostart_dir, input_shape)
all_images, all_filenames = load_images_from_directory(all_dir, input_shape)

if len(lostart_images) == 0 or len(all_images) == 0:
    print("Error: No images loaded. Check directories.")
    exit()

lostart_embeddings = base_network.predict(lostart_images)
all_embeddings = base_network.predict(all_images)

# Clustering
num_clusters = 50
kmeans = KMeans(n_clusters=num_clusters, random_state=0, n_init="auto")
kmeans.fit(lostart_embeddings)
cluster_centers = kmeans.cluster_centers_
lostart_cluster_labels = kmeans.labels_

# Save cluster images
save_cluster_images(lostart_dir, lostart_filenames, lostart_cluster_labels, output_base_dir)

# Save query results (with combined plots)
save_query_results(all_dir, all_filenames, all_embeddings,
                   lostart_dir, lostart_filenames, lostart_cluster_labels,
                   cluster_centers, output_base_dir)

print(f"Processing complete. Results saved to: {output_base_dir}")

Error loading image 00548.jpg: cannot identify image file <_io.BytesIO object at 0x32665cb30>
69/69 ━━━━━━━━━━━━━━━━━━━━ 34s 488ms/step
35/35 ━━━━━━━━━━━━━━━━━━━━ 18s 507ms/step
Processing complete. Results saved to: Exports/2025_03_26-Untrained Model First Run
